# Twitter Sentiment Analysis

## Problem Statement

Twitter generates millions of posts daily. Manually analyzing sentiment is impossible at scale.

This project automatically classifies tweets as **Positive** or **Negative** using NLP and Logistic Regression on the Sentiment140 dataset (1.6M tweets).

## Importing Dataset

In [1]:
!pip install kaggle

In [2]:
!kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:00<00:00, 94.1MB/s]



In [3]:
import zipfile
with zipfile.ZipFile("sentiment140.zip", "r") as zip:
    zip.extractall("sentiment140")
print("Dataset extracted succesfully.")

Dataset extracted succesfully.


## Importing Libraries

In [4]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [5]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Data Processing

In [6]:
column_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv("sentiment140/training.1600000.processed.noemoticon.csv", names= column_names, encoding="ISO-8859-1")

In [7]:
twitter_data.shape

(1600000, 6)

In [8]:
twitter_data.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [9]:
twitter_data.replace({'target':{4:1}}, inplace=True)

In [10]:
twitter_data.isnull().sum()

,0
target,0
ids,0
date,0
flag,0
user,0
text,0


In [11]:
twitter_data['target'].value_counts()

,count
target,
0,800000
1,800000


## Stemming

In [12]:
port_stem = PorterStemmer()

In [13]:
def stemming(content):

    stemmed_content = re.sub('[^a-zA-Z]',' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)

    return stemmed_content

In [14]:
twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming)

In [15]:
twitter_data.head()

,target,ids,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


In [16]:
X = twitter_data['stemmed_content'].values
Y = twitter_data['target'].values

## Train Test Splitting

In [17]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [18]:
vectorizer = TfidfVectorizer()  # Converting text to TF-IDF features

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

## Model Training

In [19]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, Y_train)

LogisticRegression(max_iter=1000)

## Model Evaluation

In [20]:
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)

In [21]:
print('Accuracy score on the training data :', training_data_accuracy)

Accuracy score on the training data : 0.79871953125


In [22]:
X_test_prediction = model.predict(X_test)
testing_data_accuracy = accuracy_score(Y_test, X_test_prediction)

In [23]:
print('Accuracy score on the testing data :', testing_data_accuracy)

Accuracy score on the testing data : 0.77668125


## Sample Predictions

In [24]:
X_new = X_test[12000]
print(Y_test[12000])

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
    print('Negative Tweet')
else:
    print('Positive Tweet')

1
[1]
Positive Tweet


## Conclusion

Successfully classified tweet sentiment with **81% training accuracy** and **77.8% test accuracy** using TF-IDF + Logistic Regression.

- Stemming and stopword removal improved text quality
- Future scope: BERT, real-time Twitter API, Flask deployment